# Tillicum Slurm Ops

A guide to setting up SSH, managing persistent slurm jobs, and working with tillicum remotely.

## 1. SSH Config Setup

The `ssh_config_templates/` directory contains two files that need to go into your `~/.ssh/`:

- **`config`** - Main SSH config with hosts for `klone-login`, `klone-node`, `tillicum-login`, and `moana`. Uses `ControlMaster` for persistent connections so you only authenticate once.
- **`klone-node-config`** - Included by the main config for the `klone-node` host. Contains the `ProxyJump` setup so you can SSH directly to a compute node through the login node.

**Before copying**, edit `ssh_config_templates/config` and replace `deanlcs` with your own username.

Then copy them into place:

In [ ]:
! cp ssh_config_templates/* ~/.ssh/

## 2. First Login to Tillicum

Test that your SSH config works. The first time you connect you'll need to authenticate (2FA, password, etc). After that, `ControlMaster` keeps the connection alive so subsequent SSH commands reuse it without re-authenticating.

In [ ]:
! ssh tillicum-login echo "Connected successfully"

## 3. Copy the Job Controller Script

`job_controller.sh` manages a persistent interactive slurm job named `proxy_jump`. It either:
- **Starts** a new interactive job if none is running
- **Reports** the node and remaining time if one already exists

Copy it to the remote host:

In [ ]:
! scp job_controller.sh tillicum-login:~/

## 4. Start the Job Controller

This SSHs into tillicum, creates (or attaches to) a tmux session called `job_controller`, and runs the script inside it.

- If no `proxy_jump` job exists, it starts one interactively via `srun`
- If one already exists, it prints the node and time remaining
- The tmux session persists even if you disconnect

**Note:** You may want to add resource flags to the `srun` command in `job_controller.sh` (e.g. `--qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00`).

In [ ]:
! ssh tillicum-login -t 'tmux new-session -A -s job_controller "bash ~/job_controller.sh; bash"'

## 5. Send Commands Remotely

Once the tmux session is running, you can send commands to it without attaching. This is useful for scripting or running things from your notebook.

In [ ]:
# Send a command to the running tmux session (runs inside the job)
! ssh tillicum-login 'tmux send-keys -t job_controller "hostname" Enter'

In [ ]:
# Capture the latest output from the tmux session
! ssh tillicum-login 'tmux capture-pane -t job_controller -p' | tail -20

## 6. Check Job Status / Get a Terminal

You can check the job status without tmux, or attach interactively to get a full terminal.

In [ ]:
# Check job status remotely (no tmux needed)
! ssh tillicum-login 'squeue --me --name=proxy_jump --format="%i %N %L %T" --noheader'

To get an interactive terminal, run this in a regular terminal (not the notebook):

```bash
# Attach to the tmux session (gives you a full terminal on the compute node)
ssh tillicum-login -t 'tmux attach -t job_controller'
```

To detach from tmux without killing it: press `Ctrl-b` then `d`.

## 7. Useful Commands Reference

```bash
# Cancel the running job
ssh tillicum-login 'scancel --name=proxy_jump'

# Kill the tmux session
ssh tillicum-login 'tmux kill-session -t job_controller'

# List all your running jobs
ssh tillicum-login 'squeue --me'

# Port forward from the compute node (e.g. for vllm on port 8555)
# First get the node name from squeue, then:
ssh tillicum-login -L 8555:<NODE>:8555
```

## Tillicum old

Get uv and hf on tillicum

```bash

# make your own scratch dir

mkdir /gpfs/scrubbed/deanlcs
mkdir /gpfs/scrubbed/deanlcs/cache
chmod 700 /gpfs/scrubbed/deanlcs

# symlink home dir cache to scratch dir
ln -s /gpfs/scrubbed/deanlcs/cache ~/.cache


# download gh
VERSION="2.86.0"  # Set your desired version
ARCH="amd64"      # or "arm64" for ARM systems

mkdir -p ~/.local/bin && \
cd /tmp && \
wget -q "https://github.com/cli/cli/releases/download/v${VERSION}/gh_${VERSION}_linux_${ARCH}.tar.gz" -O gh.tar.gz && \
tar -xzf gh.tar.gz && \
cp gh_${VERSION}_linux_${ARCH}/bin/gh ~/.local/bin/ && \
chmod +x ~/.local/bin/gh && \
rm -rf gh_${VERSION}_linux_${ARCH} gh.tar.gz && \
echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc && \
source ~/.bashrc && \
gh --version

# go make a personal access token here https://github.com/settings/tokens
# run auth to login
gh auth login


# clone some repo
git clone https://github.com/SafeDesign-ai/eval-bench


# when starting a new job, we need to load cuda by running
salloc --qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00
module load gcc/13.4.0
module load cuda/13.0.0


# set the UV_CACHE and UV project env env vars in the env file the uv project is looking at
export UV_PROJECT_ENVIRONMENT="/gpfs/scrubbed/deanlcs/cache/uv/eval_bench"
export UV_CACHE_DIR="/gpfs/scrubbed/deanlcs/cache/uv"
source /gpfs/scrubbed/deanlcs/cache/uv/eval_bench/bin/activate


shh tillicum.hyak.uw.edu -L 8555:g001.hyak.local:8555

# uv "reduced performance" / different-filesystem warning:
# uv wants cache and the venv on the *same* filesystem so it can hardlink from cache into the env.
# If they're on different mounts (e.g. home vs scratch), it falls back to full copies and warns.
# Fix 1 (best): put Python install on scratch too so cache + env + Python are all same FS:
export UV_PYTHON_INSTALL_DIR="/gpfs/scrubbed/deanlcs/cache/uv/python"
# Then run once: uv python install 3.12   (or whatever version) so it installs there.
#
# Fix 2 (if you can't get same FS): tell uv to not use hardlinks; avoids the fallback surprise.
# export UV_LINK_MODE=copy    # slower but predictable
# export UV_LINK_MODE=symlink # works across FS; env breaks if cache is moved/deleted

# TODO from here, make an ondemand vscode server and setup vllm on eval-bench with it 
```

## hyak old

```bash

# login to main node
ssh klone-login

# open a tmux window to avoid detach bugs
tmux

# hyak
# request an interactive job for 4 hours
salloc --account=argon --partition=gpu-l40 --gpus=1 --mem=64G --time=4:00:00 --job-name=vsc-proxy-jump


# tillicum
salloc --qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00



# on a new terminal 
# change your hyak node ssh config by running
bash ~/.ssh/set-hyak-node.sh

# then ssh to klone-node
ssh klone-node

# now you can use proxyjump extension (Remote ssh to klone node)


# if you want to find the jobs you are running
squeue --user $USER

# and to cancel a job
scancel JOBID

```